concatinate long format of different data sources to one combined df
manually align the df structure from different sources 

In [1]:
import statsmodels.formula.api as smf
import pandas as pd          
import matplotlib.pyplot as plt 
import numpy as np
from legacy_pipe.legacy_pipe.my_functions import extract_birth_year_from_dob
from tqdm import tqdm
import seaborn as sns

# HardDrive_thirties
HardDrive_thirties_long_format = pd.read_csv("/home/gaia/Projects/legacy_data/refactored_project/data/processed/HardDrive_thirties_long_format.csv")
# HardDrive_thirties_metadata = pd.read_csv("/home/gaia/Projects/legacy_data/refactored_project/data/processed/HardDrive_thirties_full_metadata_filtered.csv")
HardDrive_thirties_long_format['source'] = 'HardDrive_thirties'

HardDrive_over_thirties_long_format = pd.read_csv("/home/gaia/Projects/legacy_data/refactored_project/data/processed/HardDrive_over_thirties_long_format.csv")
# HardDrive_over_thirties_metadata = pd.read_csv("/home/gaia/Projects/legacy_data/refactored_project/data/processed/HardDrive_over_thirties_full_metadata_filtered.csv")
HardDrive_over_thirties_long_format['source'] = 'HardDrive_over_thirties'

HardDrive_under_thirties_long_format = pd.read_csv("/home/gaia/Projects/legacy_data/refactored_project/data/processed/HardDrive_under_thirties_long_format.csv")
# HardDrive_under_thirties_metadata = pd.read_csv("/home/gaia/Projects/legacy_data/refactored_project/data/processed/HardDrive_under_thirties_full_metadata_filtered.csv")
HardDrive_under_thirties_long_format['source'] = 'HardDrive_under_thirties'

# add AllTheRages 30to35 to combined_df
drive_30to35 = pd.read_csv("/home/gaia/Projects/legacy_data/refactored_project/data/processed/drive_30to35_long_format.csv")
drive_30to35['source'] = 'drive_30to35'

snbb = pd.read_pickle('/home/gaia/Projects/Gaia_gm_vol_cleaned_with_unique_id.pkl').reset_index(drop=False)
snbb['source'] = 'snbb'
snbb.shape
print(snbb.columns)


print(f"amount of unique region labels: {snbb['index'].nunique()}")
snbb_unique_indexes = snbb['index'].unique()

# make index column int and add 1 to each value
snbb['index'] = snbb['index'].astype(int) + 1
sample = snbb.head(1000)



ModuleNotFoundError: No module named 'legacy_pipe'

In [ ]:
# fill every cell in snbb tissue column that is missing with 'gm_volume_mm3'
snbb['tissue'] = snbb['tissue'].fillna('gm_volume_mm3')
# print amount of rows with tissue is missing 
print("Rows with missing tissue SNBB:", snbb['tissue'].isnull().sum())

snbb_sample = snbb.head(1000)

Aling GNBB

In [ ]:
gnbb_volumes = pd.read_csv('/home/gaia/Projects/gnbb.csv')

gnbb_volumes['source'] = 'gnbb_CDs'

gnbb_volumes['session_id'] = gnbb_volumes['session_id'].astype(str)

# remove 'ses-' from session_id column in gnbb_volumes
gnbb_volumes['session_id'] = gnbb_volumes['session_id'].str.replace('ses-', '')

# remove 'sub-' from subject_id column in gnbb_volumes
gnbb_volumes['subject_id'] = gnbb_volumes['subject_id'].str.replace('sub-', '')

# Drop duplicated key columns after merge
gnbb_volumes.drop(columns=['unique_id', 'scan_date'], inplace=True, errors='ignore')

gnbb_volumes['birth_year'] = gnbb_volumes['dob'].apply(extract_birth_year_from_dob)



Align region label to be only the index

In [ ]:
print("GNBB Volumes DataFrame Columns:")
print(gnbb_volumes.columns)

print("SNBB column names:")
print(snbb.columns)

# remove the string "schaefer2018tian2020_400_7_" from the region_label column in gnbb_volumes
gnbb_volumes['region_label'] = gnbb_volumes['region_label'].str.replace('schaefer2018tian2020_400_7_', '')

drive_30to35['region_label'] = drive_30to35['region_label'].str.replace('schaefer2018tian2020_400_7_', '')

HardDrive_thirties_long_format['region_label'] = HardDrive_thirties_long_format['region_label'].str.replace('schaefer2018tian2020_400_7_', '')

HardDrive_over_thirties_long_format['region_label'] = HardDrive_over_thirties_long_format['region_label'].str.replace('schaefer2018tian2020_400_7_', '')

HardDrive_under_thirties_long_format['region_label'] = HardDrive_under_thirties_long_format['region_label'].str.replace('schaefer2018tian2020_400_7_', '')


Align SNBB

In [ ]:
# drop the following columns from snbb: name, base_name, Label Name, network, component, hemisphere, metric
snbb = snbb.drop(columns=['name', 'base_name', 'Label Name', 'network', 'component', 'hemisphere', 'metric'], errors='ignore')

# drop the following clumns from gnbb_volumes: atlas_name, 'csf_volume_cm3', 'wm_volume_cm3'
gnbb_volumes = gnbb_volumes.drop(columns=['atlas_name', 'csf_volume_cm3', 'wm_volume_cm3'], errors='ignore')

# Define SNBB → GNBB column mapping
snbb_to_gnbb = {
    "unique_id": "subject_id",
    "session_id": "session_id",
    "protocol_name": "protocol",
    "sex": "sex",
    "dob": "dob",
    "birth_year": "birth_year",
    "volume": "volume_mm3",
    "tissue": "tissue_type",
    "age_at_scan": "age_in_years",
    "index": "region_label", 
    "tiv": "tiv",
    "total_gm_volume" : "gm_volume_cm3", 
    "source": "source"
}

# Rename SNBB columns
snbb = snbb.rename(columns=snbb_to_gnbb)

# Get all columns from GNBB
gnbb_cols = gnbb_volumes.columns

# Align SNBB columns to GNBB schema (add missing ones as NaN)
snbb = snbb.reindex(columns=gnbb_cols)
print(f"snbb columns are: {snbb.columns}")
snbb_head = snbb.head(100)

snbb.describe()



### fill institute and manufacturer for snbb 
Scanner link - https://mri.tau.ac.il/Human_3T_Prisma

In [ ]:
# is there any value that is not null in the institute column of snbb
not_missing_institues = snbb[snbb['institute'].notna()]['institute'].unique()
print(f"Institutes not missing in SNBB: {not_missing_institues}")

# fill institute with 'Tel-Aviv University' 
snbb['institute'] = snbb['institute'].fillna('Tel-Aviv University')

not_missing_manufacturers = snbb[snbb['manufacturer'].notna()]['manufacturer'].unique()
print(f"Manufacturers not missing in SNBB: {not_missing_manufacturers}")

# fill manufacturer with 'SIEMENS'
snbb['manufacturer'] = snbb['manufacturer'].fillna('SIEMENS')

# validate no missing values in institute and manufacturer columns
print("Rows with missing institute SNBB:", snbb['institute'].isnull().sum())
print("Rows with missing manufacturer SNBB:", snbb['manufacturer'].isnull().sum())

In [ ]:
# Concatenate
combined_df = pd.concat([gnbb_volumes, snbb, drive_30to35, HardDrive_thirties_long_format, HardDrive_over_thirties_long_format, HardDrive_under_thirties_long_format], ignore_index=True)

print("Combined shape:", combined_df.shape)
print("Columns:", combined_df.columns.tolist())
print(f"amount of unique region labels: {combined_df['region_label'].nunique()}")
print(f"amount of unique subject ids: {combined_df['subject_id'].nunique()}")

#save a list of unique region labels
unique_region_labels = combined_df['region_label'].unique()
#types of values in unique_region_labels
print(f"types of values in unique_region_labels: {type(unique_region_labels[0])}")

# change all region labels to int
combined_df['region_label'] = combined_df['region_label'].astype(int)
print(f"amount of unique region labels: {combined_df['region_label'].nunique()}")

print(f"head of combined is: {combined_df.head(10)}")



In [ ]:
# keep only rows where tissue_type is 'gm_volume_mm3'
combined_df = combined_df[combined_df['tissue_type'] == 'gm_volume_mm3']

print(f"Rows with missing tissue_type combined: {combined_df['tissue_type'].isnull().sum()}")

# apply extract_birth_year_from_dob to the dob column to create a new birth_year column
combined_df['birth_year'] = combined_df['dob'].apply(extract_birth_year_from_dob)

# # remove duplications based on subject_id , session_id, region_label, tissue_type
# combined_df = combined_df.drop_duplicates(subset=['subject_id', 'session_id', 'region_label', 'tissue_type'])


In [ ]:
# rows with missing institute
missing_institute_rows = combined_df[combined_df['institute'].isnull()]
print(f"Rows with missing institute in combined_df: {missing_institute_rows.shape[0]}")

# unique institutes in combined_df
unique_institutes = combined_df['institute'].unique()
print(f"Unique institutes in combined_df: {unique_institutes}")

# replace 'Tel Aviv Medical Center' and 'TEL AVIV MEDICAL CENTER' with 'ICHILOV TEL AVIV'
combined_df['institute'] = combined_df['institute'].replace(['Tel Aviv Medical Center', 'TEL AVIV MEDICAL CENTER'], 'ICHILOV TEL AVIV')

# tests
to make sure all important info is there

In [ ]:
# rows without volume_mm3
rows_without_volume = combined_df[combined_df['volume_mm3'].isnull()]
print(f"Rows without volume_mm3: {rows_without_volume.shape[0]}")

# rows without 'sex', 'age_in_years', 'dob', 'birth_year'
rows_missing_important_info = combined_df[
    combined_df['sex'].isnull() |
    combined_df['age_in_years'].isnull() |
    combined_df['dob'].isnull() |
    combined_df['birth_year'].isnull()
]
print(f"Rows missing important info: {rows_missing_important_info.shape[0]}")

# remove rows with missing important info from combined_df
combined_df = combined_df.drop(rows_missing_important_info.index)

# make sure it worked 
rows_missing_important_info_after = combined_df[
    combined_df['sex'].isnull() |
    combined_df['age_in_years'].isnull() |
    combined_df['dob'].isnull() |
    combined_df['birth_year'].isnull()
]
print(f"Rows missing important info after removal: {rows_missing_important_info_after.shape[0]}")

In [ ]:
# save combined_df to a pickle file
# combined_df.to_pickle('/home/gaia/Projects/legacy_data/combined_gm_volumes.pkl')

move this to a dashboard with descriptive statistics  

In [ ]:
# # amount of 30 year old unique subjects snbb 
# snbb_thirties = snbb[(snbb['age_in_years'] >= 30) & (snbb['age_in_years'] < 36)]['subject_id'].nunique()
# print(f"amount of 30 year old unique subjects snbb: {snbb_thirties}")
# # print birth year range in snbb_thirties:
# snbb_thirties_birth_years = snbb[(snbb['age_in_years'] >= 30) & (snbb['age_in_years'] < 36)]['birth_year']
# print(f"birth year range in snbb_thirties: {snbb_thirties_birth_years.min()} - {snbb_thirties_birth_years.max()}")

# # amount of 30 year old unique subjects gnbb 
# gnbb_thirties = gnbb_volumes[(gnbb_volumes['age_in_years'] >= 30) & (gnbb_volumes['age_in_years'] < 36)]['subject_id'].nunique()
# print(f"amount of 30 year old unique subjects gnbb: {gnbb_thirties}")


# # print birth year range in gnbb_thirties:
# gnbb_thirties_birth_years = gnbb_volumes[(gnbb_volumes['age_in_years'] >= 30) & (gnbb_volumes['age_in_years'] < 36)]['birth_year']
# print(f"birth year range in gnbb_thirties: {gnbb_thirties_birth_years.min()} - {gnbb_thirties_birth_years.max()}")

# # amount of 30 year old unique subjects HardDrive_thirties 
# HardDrive_thirties = HardDrive_thirties_long_format[(HardDrive_thirties_long_format['age_in_years'] >= 30) & (HardDrive_thirties_long_format['age_in_years'] < 36)]['subject_id'].nunique()
# print(f"amount of 30 year old unique subjects HardDrive_thirties: {HardDrive_thirties}")
# # print birth year range in HardDrive_thirties:
# HardDrive_thirties_birth_years = HardDrive_thirties_long_format[(HardDrive_thirties_long_format['age_in_years'] >= 30) & (HardDrive_thirties_long_format['age_in_years'] < 36)]['birth_year']
# print(f"birth year range in HardDrive_thirties: {HardDrive_thirties_birth_years.min()} - {HardDrive_thirties_birth_years.max()}")
